In [ ]:
import json
import time
import feedparser
import pandas as pd
import ollama

Configuration

In [ ]:
MODEL = "qwen2.5:3b"
BATCH_SIZE = 5          # start small for reliability
SLEEP_BETWEEN_CALLS = 1

FEEDS = {
    "Health": "https://feeds.bbci.co.uk/news/health/rss.xml",
    "Business": "https://feeds.bbci.co.uk/news/business/rss.xml",
    "Technology": "https://feeds.bbci.co.uk/news/technology/rss.xml",
    "Science": "https://feeds.bbci.co.uk/news/science_and_environment/rss.xml?edition=uk"
}

Scrape the RSS feeds

In [ ]:
rows = []

for category, url in FEEDS.items():
    print(f"Loading feed: {category}")

    feed = feedparser.parse(url)

    for entry in feed.entries:
        rows.append({
            "source": "BBC",
            "feed_category": category,
            "title": entry.get("title", "").strip(),
            "published": entry.get("published", "").strip(),
            "link": entry.get("link", "").strip(),
            "description": entry.get("summary", "").strip()
        })

df = pd.DataFrame(rows)

# Remove duplicates
df.drop_duplicates(subset=["link"], inplace=True)
df.reset_index(drop=True, inplace=True)

print("\nRaw rows collected:", len(df))

# Save raw file
df.to_csv("bbc_news_raw.csv", index=False)

AI Classification

In [ ]:
def classify_batch(headlines):
    """
    Classify a batch of headlines using local Ollama model.
    Returns JSON list.
    """

    numbered = "\n".join(
        f"{i+1}. {headline}" for i, headline in enumerate(headlines)
    )

    prompt = f"""
For each headline below, return ONLY a valid JSON array.

Each object must contain:
index
topic
sentiment
keywords

Rules:
- index = matching number
- topic = short category
- sentiment = positive, neutral, or negative
- keywords = list of exactly 3 short keywords
- No explanation
- No markdown
- JSON only

Example:
[
  {{
    "index": 1,
    "topic": "Technology",
    "sentiment": "neutral",
    "keywords": ["AI", "chips", "market"]
  }}
]

Headlines:
{numbered}
"""

    response = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    text = response["message"]["content"].strip()

    # Remove code fences if model adds them
    text = text.replace("```json", "").replace("```", "").strip()

    return json.loads(text)

# Add output columns
df["ai_topic"] = ""
df["sentiment"] = ""
df["keywords"] = ""

titles = df["title"].tolist()

for start in range(0, len(titles), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(titles))
    batch = titles[start:end]

    print(f"Processing rows {start} to {end - 1}")

    try:
        results = classify_batch(batch)

        for item in results:
            idx = start + int(item["index"]) - 1

            if idx < len(df):
                df.at[idx, "ai_topic"] = item.get("topic", "")
                df.at[idx, "sentiment"] = item.get("sentiment", "")
                df.at[idx, "keywords"] = ", ".join(item.get("keywords", []))

    except Exception as e:
        print("Batch failed:", e)

    time.sleep(SLEEP_BETWEEN_CALLS)

In [1]:
"""
bbc_rss_ollama_scraper.py

Purpose:
- Scrape BBC RSS feeds
- Use Ollama (local AI model) to classify headlines
- Save final CSV files

Install:
pip install feedparser pandas ollama

Before Running:
1. Make sure Ollama desktop app/service is running
2. Pull a model in terminal:
   ollama pull qwen2.5:3b
"""


# =========================
# CONFIG
# =========================



# =========================
# STEP 1: SCRAPE RSS FEEDS
# =========================



# =========================
# STEP 2: AI CLASSIFICATION
# =========================



# =========================
# STEP 3: SAVE FINAL FILE
# =========================



Loading feed: Health
Loading feed: Business
Loading feed: Technology
Loading feed: Science

Raw rows collected: 151
Processing rows 0 to 4
Processing rows 5 to 9
Processing rows 10 to 14
Processing rows 15 to 19
Processing rows 20 to 24
Processing rows 25 to 29
Processing rows 30 to 34
Processing rows 35 to 39
Processing rows 40 to 44
Processing rows 45 to 49
Processing rows 50 to 54
Processing rows 55 to 59
Processing rows 60 to 64
Processing rows 65 to 69
Processing rows 70 to 74
Processing rows 75 to 79
Processing rows 80 to 84
Processing rows 85 to 89
Processing rows 90 to 94
Processing rows 95 to 99
Processing rows 100 to 104
Processing rows 105 to 109
Processing rows 110 to 114
Processing rows 115 to 119
Processing rows 120 to 124
Processing rows 125 to 129
Processing rows 130 to 134
Processing rows 135 to 139
Processing rows 140 to 144
Processing rows 145 to 149
Processing rows 150 to 150

Done.
Rows: 151
Saved files:
- bbc_news_raw.csv
- bbc_news_ai.csv

Preview:
  source feed_

Run the program

In [ ]:
df.to_csv("bbc_news_ai.csv", index=False)

print("\nDone.")
print("Rows:", len(df))
print("Saved files:")
print("- bbc_news_raw.csv")
print("- bbc_news_ai.csv")
print("\nPreview:")
print(df.head(10))

In [2]:
df.head()

,source,feed_category,title,published,link,description,ai_topic,sentiment,keywords
0,BBC,Health,Don't feel like exercising? Maybe it's the wro...,"Tue, 14 Apr 2026 23:54:08 GMT",https://www.bbc.com/news/articles/cd6lzpxwx50o...,"Time your workout to your body clock, health r...",Health,neutral,"exercise, time of day"
1,BBC,Health,'I'm not being listened to' - new health plan ...,"Wed, 15 Apr 2026 10:01:48 GMT",https://www.bbc.com/news/articles/cm2kke1jn8xo...,New plans to improve healthcare for women and ...,Society,negative,"ignored, women"
2,BBC,Health,A cold could kill my daughter - hospital visit...,"Wed, 15 Apr 2026 05:57:53 GMT",https://www.bbc.com/news/articles/c145ej273evo...,"Rebecca Quayle, who has terminal cancer, has h...",Health,negative,"cold, hospital visits"
3,BBC,Health,Single-sex space guidance for organisations to...,"Tue, 14 Apr 2026 14:54:17 GMT",https://www.bbc.com/news/articles/cn08849wz07o...,Equalities minister Bridget Phillipson says el...,Society,neutral,"space guidance, organizations, elections"
4,BBC,Health,Doctors' strikes can have surprising benefits ...,"Mon, 13 Apr 2026 23:06:45 GMT",https://www.bbc.com/news/articles/cp3l2pygnlyo...,Some hospital trusts tell the BBC previous act...,Healthcare,mixed,"strikes, benefits, sustainability"
